# Hands-on Lab: Analyzing Historical Stock/Revenue Data and Building a Dashboard

Tesla and GameStop stock and revenue analysis.

In [ ]:
# Install packages if needed (run once)
# !pip install yfinance beautifulsoup4 pandas requests plotly

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Question 1: Use yfinance to Extract Tesla Stock Data

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2: Use Webscraping to Extract Tesla Revenue Data

In [ ]:
url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"
html_data = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
soup = BeautifulSoup(html_data, "html.parser")

In [ ]:
# Extract the Tesla quarterly revenue table
tables = pd.read_html(html_data, match="Tesla Quarterly Revenue")
tesla_revenue = tables[0]
tesla_revenue = tesla_revenue.rename(columns={
    tesla_revenue.columns[0]: "Date",
    tesla_revenue.columns[1]: "Revenue"
})
tesla_revenue = tesla_revenue[["Date", "Revenue"]]
tesla_revenue["Revenue"] = tesla_revenue["Revenue"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False)
tesla_revenue["Revenue"] = pd.to_numeric(tesla_revenue["Revenue"], errors="coerce")
tesla_revenue = tesla_revenue[tesla_revenue["Revenue"].notna()]
tesla_revenue.tail()

## Question 3: Use yfinance to Extract GameStop Stock Data

In [ ]:
gamestop = yf.Ticker("GME")
gme_data = gamestop.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4: Use Webscraping to Extract GameStop Revenue Data

In [ ]:
url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"
html_data = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
soup = BeautifulSoup(html_data, "html.parser")

In [ ]:
tables = pd.read_html(html_data, match="GameStop Quarterly Revenue")
gme_revenue = tables[0]
gme_revenue = gme_revenue.rename(columns={
    gme_revenue.columns[0]: "Date",
    gme_revenue.columns[1]: "Revenue"
})
gme_revenue = gme_revenue[["Date", "Revenue"]]
gme_revenue["Revenue"] = gme_revenue["Revenue"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False)
gme_revenue["Revenue"] = pd.to_numeric(gme_revenue["Revenue"], errors="coerce")
gme_revenue = gme_revenue[gme_revenue["Revenue"].notna()]
gme_revenue.tail()

## Question 5 and 6: Plot Stock Price and Revenue

In [ ]:
def make_graph(stock_data, revenue_data, title):
    stock_data = stock_data.copy()
    revenue_data = revenue_data.copy()
    stock_data["Date"] = pd.to_datetime(stock_data["Date"]).dt.tz_localize(None) if pd.api.types.is_datetime64tz_dtype(stock_data["Date"]) else pd.to_datetime(stock_data["Date"])
    revenue_data["Date"] = pd.to_datetime(revenue_data["Date"])

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Historical Share Price", "Historical Revenue"))
    fig.add_trace(go.Scatter(x=stock_data["Date"], y=stock_data["Close"], name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=revenue_data["Date"], y=revenue_data["Revenue"], name="Revenue"), row=2, col=1)
    fig.update_layout(title=title, height=700, showlegend=True)
    fig.update_yaxes(title_text="Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD millions)", row=2, col=1)
    fig.show()

In [ ]:
# Question 5: Tesla graphs
make_graph(tesla_data, tesla_revenue, "Tesla Stock Price and Revenue")

In [ ]:
# Question 6: GameStop graphs
make_graph(gme_data, gme_revenue, "GameStop Stock Price and Revenue")